
# Elite Dangerous Local Database
> Cached Elite Dangerous systems data

In [ ]:
#| default_exp eddb.localdb

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import sys, logging, typing, sqlite3, os, csv, json, gzip, subprocess
import pandas as pd

from typing import Any, NamedTuple
from contextlib import contextmanager
from edcompanion.core import configuration


In [ ]:
from confproxy.core import init_console_logging
from edcompanion.eddb.readers import dbfilereader, dbfile_process
init_console_logging(__name__)

2025-12-22T09:09:49+0100 INFO	6693	__main__	core.py	init_console_logging	40	Installed <StreamHandler stderr (INFO)> for __main__


<Logger __main__ (INFO)>

In [ ]:
#| exporti
syslog = logging.getLogger(__name__)
eddb_config = configuration["EDDB"]
syslog.info(f"Loading module {__name__}, config={eddb_config}")


2025-12-22T09:09:49+0100 INFO	6693	__main__	4091137986.py	<module>	4	Loading module __main__, config=Section EDDB in /home/fenke/.config/EDTravelCompanion/settings.ini


## SQLite

### SQLiteQueryParams

In [ ]:
#| export
class SQLiteQueryParams(NamedTuple):
    as_param: typing.Callable
    append_param: typing.Callable   
    get_params: typing.Callable  


In [ ]:
#| export

def sqlite_query_params(log=None) -> SQLiteQueryParams:
    sql_params = {}

    def as_param(name:str):
        return f":{str(name)}"
   
    def append_param(name:str, value:Any):
        assert str(name) not in sql_params, f"Duplicate parameter {name}"
        assert len(sql_params) < 32766, "SQLite does not allow more then approx. 32k bound parameters"

        last_name = str(name)
        sql_params[last_name] = value
        return as_param(last_name)
    
    def get_params():
        return sql_params.copy()
    
    return SQLiteQueryParams(
        as_param=as_param,
        append_param=append_param if log is None else lambda p: log(append_param(p)),
        get_params=get_params
    )

### SQLiteConnectionInterface

In [ ]:
#| export

class SQLiteConnectionInterface(typing.NamedTuple):
    cursor: typing.Callable
    commit: typing.Callable
    rollback: typing.Callable
    close: typing.Callable
    execute: typing.Callable
    executemany: typing.Callable


In [ ]:
sys.version_info

sys.version_info(major=3, minor=11, micro=10, releaselevel='final', serial=0)

In [ ]:
#| export
def sqllite_connection_interface(
        database:str=":memory:",
    ) -> SQLiteConnectionInterface:

    vi = sys.version_info
    assert vi.major >= 3, f"Python >= 3.0 required. Found {vi.major}.{vi.minor}.{vi.micro}"

    if vi.minor > 11:   
        syslog.info("Using legacy autocommit")
        connection = sqlite3.connect(database, autocommit=sqlite3.LEGACY_TRANSACTION_CONTROL, isolation_level='DEFERRED')
    else:
        syslog.info("Using isolation_level=None")
        connection = sqlite3.connect(database, isolation_level=None)

    def close():
        syslog.info("Closing connection")
        connection.close()

    def commit():
        syslog.info("Committing on connection")
        connection.commit()

    def rollback():
        syslog.info("Rolling back connection")
        connection.rollback()
    
    def cursor():
        syslog.info("Creating cursor")
        return connection.cursor()
    
    def execute(sql:str, params:tuple|dict=()):
        return connection.execute(sql, params)
    
    def execute_many(sql:str, params:list[tuple|dict]=[]):
        return connection.executemany(sql, params)

    syslog.info("Returning connection-interface")
    return SQLiteConnectionInterface(
        cursor=cursor,
        commit=commit,
        rollback=rollback,
        close=close,
        execute=execute,
        executemany=execute_many
    )

### Context manager

In [ ]:
#| export

@contextmanager
def sqllite_connection(*args, **kwargs):
    syslog.info(f"Opening connection-interface to {args}")
    interface = sqllite_connection_interface(*args, **kwargs)
    try:
        yield interface
    finally:
        interface.close()

### Test

In [ ]:
with sqllite_connection() as ci:
    
    ci.execute("CREATE TABLE lang(name, first_appeared)")

    # This is the named style used with executemany():
    data = (
        {"name": "C", "year": 1972},
        {"name": "Fortran", "year": 1957},
        {"name": "Python", "year": 1991},
        {"name": "Go", "year": 2009},
    )
    ci.executemany("INSERT INTO lang VALUES(:name, :year)", data)
    for r in ci.execute("SELECT * FROM lang"):
        print(r)
        
    # This is the qmark style used in a SELECT query:
    params = (1972,)

    for r in ci.execute("SELECT * FROM lang WHERE first_appeared = :year", {'year':1957}):
        print(r)



2025-12-22T09:09:49+0100 INFO	6693	__main__	1025727644.py	sqllite_connection	5	Opening connection-interface to ()
2025-12-22T09:09:49+0100 INFO	6693	__main__	3552464633.py	sqllite_connection_interface	13	Using isolation_level=None
2025-12-22T09:09:49+0100 INFO	6693	__main__	3552464633.py	sqllite_connection_interface	38	Returning connection-interface
2025-12-22T09:09:49+0100 INFO	6693	__main__	3552464633.py	close	17	Closing connection


('C', 1972)
('Fortran', 1957)
('Python', 1991)
('Go', 2009)
('Fortran', 1957)


## ED galaxy systems database

In [ ]:
eddb_config['local_data_folder'], eddb_config['main_database']

('/home/fenke/repos/EDCompanion/data', 'eddb_systems.db')

### Systems table

In [ ]:
systems_databasefile = os.path.join(eddb_config['local_data_folder'], eddb_config['main_database'])
print(systems_databasefile)

/home/fenke/repos/EDCompanion/data/eddb_systems.db


In [ ]:
#| export

def create_systems_table(systems_databasefile, tablename='systems'):

    with sqllite_connection(systems_databasefile) as conn:

        conn.execute(f"""
                DROP TABLE IF EXISTS {tablename};
            """
        )

        conn.execute(f"""
            CREATE TABLE IF NOT EXISTS {tablename} (
                id64 BIGINT NOT NULL,
                x DOUBLE PRECISION  NOT NULL,
                y DOUBLE PRECISION  NOT NULL,
                z DOUBLE PRECISION  NOT NULL,
                name TEXT NOT NULL
            ); """
        )


In [ ]:
#| export

def create_systems_indices(systems_databasefile, tablename='systems'):

    print(f"Adding indexes ...")
    with sqllite_connection(systems_databasefile) as conn:
        for sql in f"""
            CREATE INDEX IF NOT EXISTS {tablename}_x_idx ON systems (x);
            CREATE INDEX IF NOT EXISTS {tablename}_y_idx ON systems (y);
            CREATE INDEX IF NOT EXISTS {tablename}_z_idx ON systems (z); 
            CREATE INDEX IF NOT EXISTS {tablename}_name_idx ON systems (name)
        """.split(";"):
            conn.execute(sql)


In [ ]:
#| export

def remove_duplicates(systems_databasefile, tablename='systems', column='id64'):

    with sqllite_connection(systems_databasefile) as conn:
        
        print("Removing duplicates by id")

        # create temporary index
        index_name = f"{tablename}_{column}_idx"
        conn.execute(f"CREATE INDEX IF NOT EXISTS {index_name} ON {tablename} ({column})")
        conn.execute(f"""
                DELETE FROM {tablename}
                WHERE rowid NOT IN (
                    SELECT MIN(rowid)
                    FROM {tablename}
                    GROUP BY {column}
                ); """
        )

        print(f"Adding unique index {index_name} on {tablename} {column} ...")

        conn.execute(f"DROP INDEX IF EXISTS {index_name}")
        conn.execute(f"CREATE UNIQUE INDEX IF NOT EXISTS {index_name} ON {tablename} ({column})")

#### Convert / Import through .csv

In [ ]:
infile = os.path.join(eddb_config['local_dumps'], eddb_config['systems_full'])

print(infile)


/home/fenke/Downloads/systems.json.gz


In [ ]:
#| export

def convert_systems_dumpfile_to_csv(infile, chunksize=128*1024*1024 ):

    with gzip.open(infile, 'rt') as jsonfile:
        chunknr = 0

        while True:

            chunk = jsonfile.readlines(chunksize)
            if chunk:

                csv_filename = infile.replace('.json.gz', f'_{chunknr:05d}.csv')
                syslog.info(f"Writing {csv_filename}")

                with open(csv_filename, 'w', newline='') as csvfile:
                    fieldnames = ['id64', 'x', 'y', 'z', 'name']
                    writer = csv.writer(csvfile, delimiter=',',
                                            quotechar='|', quoting=csv.QUOTE_MINIMAL)

                    writer.writerow(fieldnames)

                    for line in chunk:
                        if len(line) > 4:
                            item = json.loads(line.rstrip(',\n\r '))
                            writer.writerow( [item.get('id64')]+list(item.get('coords').values())+[item.get('name')])
                

                chunknr += 1
                yield csv_filename

            else:
                break


In [ ]:
def import_systems_csv_to_sqlite(database_file, csv_filename, tablename):
    syslog.info(f"Importing {csv_filename} to {tablename}")
    subprocess.run([
        "sqlite3",
        database_file,
        ".mode csv",
        ".headers on",
        f".import {csv_filename} {tablename}"
    ])


In [ ]:
if False:
    infile = os.path.join(eddb_config['local_dumps'], eddb_config['systems_full'])

    create_systems_table(systems_databasefile)

    for csv_filename in convert_systems_dumpfile_to_csv(infile, chunksize=256*1024*1024 ):
        import_systems_csv_to_sqlite(systems_databasefile, csv_filename, 'systems')
        os.remove(csv_filename)

    remove_duplicates(systems_databasefile)
    create_systems_indices(systems_databasefile)


2025-12-22T11:16:09+0100 INFO	6693	__main__	3474119378.py	convert_dumpfile_to_csv	12	Writing /home/fenke/Downloads/systems_00000.csv
2025-12-22T11:16:15+0100 INFO	6693	__main__	1668729119.py	import_csv_to_sqlite	2	Importing /home/fenke/Downloads/systems_00000.csv to systems
2025-12-22T11:16:17+0100 INFO	6693	__main__	3474119378.py	convert_dumpfile_to_csv	12	Writing /home/fenke/Downloads/systems_00001.csv
2025-12-22T11:16:23+0100 INFO	6693	__main__	1668729119.py	import_csv_to_sqlite	2	Importing /home/fenke/Downloads/systems_00001.csv to systems
2025-12-22T11:16:26+0100 INFO	6693	__main__	3474119378.py	convert_dumpfile_to_csv	12	Writing /home/fenke/Downloads/systems_00002.csv
2025-12-22T11:16:32+0100 INFO	6693	__main__	1668729119.py	import_csv_to_sqlite	2	Importing /home/fenke/Downloads/systems_00002.csv to systems
2025-12-22T11:16:34+0100 INFO	6693	__main__	3474119378.py	convert_dumpfile_to_csv	12	Writing /home/fenke/Downloads/systems_00003.csv
2025-12-22T11:16:40+0100 INFO	6693	__main_

#### Update systems

In [ ]:
if True:
    infile = os.path.join(eddb_config['local_dumps'], eddb_config['systems_1week'])

    create_systems_table(systems_databasefile, 'temp_systems')

    for csv_filename in convert_systems_dumpfile_to_csv(infile, chunksize=256*1024*1024 ):
        import_systems_csv_to_sqlite(systems_databasefile, csv_filename, 'temp_systems')
        os.remove(csv_filename)
        
if False:
    remove_duplicates(systems_databasefile, 'temp_systems', 'id64')

    with sqllite_connection(systems_databasefile) as conn:
        conn.execute("""
            INSERT INTO systems SELECT * FROM temp_systems ON CONFLICT (id64) DO NOTHING; 
        """)



2025-12-22T12:49:45+0100 INFO	6693	__main__	1025727644.py	sqllite_connection	5	Opening connection-interface to ('/home/fenke/repos/EDCompanion/data/eddb_systems.db',)
2025-12-22T12:49:45+0100 INFO	6693	__main__	3552464633.py	sqllite_connection_interface	13	Using isolation_level=None
2025-12-22T12:49:45+0100 INFO	6693	__main__	3552464633.py	sqllite_connection_interface	38	Returning connection-interface
2025-12-22T12:50:37+0100 INFO	6693	__main__	3552464633.py	close	17	Closing connection
2025-12-22T12:50:38+0100 INFO	6693	__main__	1148725018.py	convert_systems_dumpfile_to_csv	14	Writing /home/fenke/Downloads/systems_1week_00000.csv
2025-12-22T12:50:42+0100 INFO	6693	__main__	2520103369.py	import_systems_csv_to_sqlite	2	Importing /home/fenke/Downloads/systems_1week_00000.csv to temp_systems


### Star types

In [ ]:
main_sequence = {s:c for s, c in zip('OBAFGKMN', range(9))}
#print(json.dumps(classifications, indent=2))


In [ ]:

star_types={}
star_types_file = os.path.join(eddb_config['local_data_folder'], 'star_types.json')
print(f"File with star types: {star_types_file}")


File with star types: /home/fenke/repos/EDCompanion/data/star_types.json


In [ ]:

if not os.path.exists(star_types_file):
    print("Extracting star types")
    skipped = 0
    updated = 0

    for item in dbfilereader(os.path.join(eddb_config['local_dumps'], eddb_config['systems_1month'])):
        main_star = item.get('mainStar')

        if main_star and not star_types.get(main_star):
            print(f"updating with {main_star}")
            updated += 1
            star_types[main_star] = len(star_types)
        else:
            skipped += 1


    print(f"Updated: {updated}, skipped {skipped}")


    main_star_types = {}
    for name in set(star_types.keys()):
        first, *rest = name.split(' ')
        if len(first) == 1 and first in main_sequence:
            main_star_types[name] = len(main_star_types)

    for name in star_types:
        if name not in main_star_types:
            main_star_types[name] = len(main_star_types)

    try:
        with open(star_types_file,'wt') as jsonfile:
            json.dump(main_star_types, jsonfile, indent=3)
    except:
        pass

    star_types = main_star_types.copy()
    
else:
    with open(star_types_file,'rt') as jsonfile:
        star_types.update(json.load(jsonfile))

#print(json.dumps(main_star_types, indent=2))


In [ ]:
main_sequence

{'O': 0, 'B': 1, 'A': 2, 'F': 3, 'G': 4, 'K': 5, 'M': 6, 'N': 7}

In [ ]:
os.path.join(eddb_config['local_dumps'], eddb_config['systems_1week'])

'/home/fenke/repos/EDCompanion/data/systems_1week.json.gz'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()